# 01 · Chunking: where to cut the document?

RAG retrieves **fragments**, not entire files. We will split an authored Acme scientific note into word windows, paragraphs, sections, and segments defined by topic changes. We will then ask the same question of each variant. This notebook is self-contained and uses only a local embedding model.

After the lesson, explain why length, overlap, and section boundaries affect retrieval relevance, cost, and citation quality. Source: `SCI-FORECAST-2011:v1`, a fictional methodological note. Embeddings measure similarity; they do not prove that content is true.

In [ ]:
# Only cell with imports and project path.
import re
import sys
from pathlib import Path

import numpy as np

ROOT = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / "models.toml").exists())
sys.path.insert(0, str(ROOT / "notebooks" / "rag"))
from lab import embed_texts, load_document, section_records

## 1. Get to know the document

Metadata and version come from the file, not from the model's response. Pay attention to the part about the zero denominator: the retriever will have to find it later.

In [ ]:
doc = load_document("SCI-FORECAST-2011")
raw_body = doc["body"]
# Evaluation questions are not evidence: all strategies receive the same source text.
body = raw_body.split("## Questions for retrieval evaluation", 1)[0].strip()
sections = section_records(("SCI-FORECAST-2011",))
[(row["heading"], len(row["text"].split())) for row in sections]

## 2. Fixed windows, paragraphs, and sections

An 80-word window provides a more precisely targeted fragment, but it easily cuts off a definition from its qualification. A 250-word window carries more context but consumes more space in the prompt. Overlap copies the end of the previous fragment; it increases index costs and can multiply nearly identical hits. We are counting **words** here for readability; the model's context limit refers to **tokens**, so in a real pipeline, use the specific model's tokenizer.

In [ ]:
def word_windows(text, size, overlap=0):
    words = text.split()
    step = size - overlap
    if step <= 0:
        raise ValueError("overlap must be smaller than size")
    return [" ".join(words[start:start + size]) for start in range(0, len(words), step)]


small = word_windows(body, 80, 10)
large = word_windows(body, 250, 30)
no_overlap = word_windows(body, 80, 0)
wide_overlap = word_windows(body, 80, 30)
paragraphs = [part.strip() for part in re.split(r"\n\s*\n", body) if len(part.split()) > 8]
by_section = [row["text"] for row in sections]
[(name, len(parts), round(np.mean([len(p.split()) for p in parts]))) for name, parts in
 [("80 words", small), ("250 words", large), ("paragraph", paragraphs), ("section", by_section)]]

The first two windows can intersect a paragraph. Compare them with a section that preserves the heading and the topic boundary. At the same length of 80 words, a larger overlap creates more fragments and repeats more words in the index.

In [ ]:
{"window_boundary": (small[0][-100:], small[1][:100]), "section_heading": sections[1]["heading"],
 "overlap_cost": [(name, len(parts), sum(len(part.split()) for part in parts)) for name, parts in
                     [("0 words", no_overlap), ("10 words", small), ("30 words", wide_overlap)]]}

## 3. Semantic Chunking

Each paragraph is encoded separately. A low cosine similarity between neighbors suggests a change in topic. The threshold is a **document-dependent heuristic**: the boundary graph must be evaluated manually. We do not assume that the model knows the file structure or the business logic.

In [ ]:
vectors = embed_texts(paragraphs)
unit = vectors / np.linalg.norm(vectors, axis=1, keepdims=True)
adjacent = np.sum(unit[:-1] * unit[1:], axis=1)
cutoff = float(np.quantile(adjacent, 0.25))
boundaries = [i + 1 for i, score in enumerate(adjacent) if score < cutoff]
[(i, round(float(adjacent[i - 1]), 3), paragraphs[i][:65]) for i in boundaries]

In [ ]:
semantic = []
start = 0
for stop in [*boundaries, len(paragraphs)]:
    semantic.append("\n\n".join(paragraphs[start:stop]))
    start = stop
[(len(chunk.split()), chunk[:80]) for chunk in semantic]

A semantic boundary does not limit size on its own. If a segment becomes too long, add a maximum token limit and split it again, preferably at a paragraph break. A very short segment can be merged with its neighbor, while preserving the section ID and origin.

## 4. The same question, several ways of dividing

The query and all fragments pass through the **same** embedding model. The result is an exact cosine ranking in a small corpus; do not confuse it with generating an answer. The column `top1_ma_licznik_i_zero` checks only two words. It does not replace a manual assessment of whether a fragment is sufficient to answer the question.

Before the split, we excluded the final list of **evaluation questions**. Otherwise, a question might win the ranking even if it does not contain the answer, and the compared strategies would receive different source material.

In [ ]:
query = "Why can percentage forecast error be undefined for zero demand?"
q = embed_texts([query])[0]
q = q / np.linalg.norm(q)
results = []
for name, chunks in [("80 / overlap 0", no_overlap), ("80 / overlap 10", small),
                     ("80 / overlap 30", wide_overlap), ("250 words", large),
                     ("section", by_section), ("semantic", semantic)]:
    matrix = embed_texts(chunks)
    matrix = matrix / np.linalg.norm(matrix, axis=1, keepdims=True)
    scores = matrix @ q
    best = int(np.argmax(scores))
    results.append({"strategy": name, "chunks": len(chunks), "words_in_top1": len(chunks[best].split()),
                    "cosinus_top1": round(float(scores[best]), 3),
                    "top1_has_numerator_and_zero": "zero" in chunks[best].lower() and "percent" in chunks[best].lower(),
                    "start": chunks[best][:100]})
results

Compare the `cosinus_top1` result with the column indicating the presence of two words. A high cosine similarity might refer to a related topic, while overlap can both save a sentence at the edge of a window and create additional, nearly identical hits. Only reading the top-1 will confirm whether the fragment contains the full answer.

## 5. Parent → child hierarchy

We index short children, but upon a hit, we can return the parent section. In production, `child_id`, `parent_id`, version, and text hash must be stored. The parent is sometimes too long for the prompt: in such cases, only adjacent children can be retrieved or the section can be shortened **after** the evidence is evaluated.

In [ ]:
children = [(row["id"], i, child) for row in sections for i, child in enumerate(word_windows(row["text"], 80, 10))]
child_vectors = embed_texts([text for _, _, text in children])
child_vectors /= np.linalg.norm(child_vectors, axis=1, keepdims=True)
best_child = children[int(np.argmax(child_vectors @ q))]
parent = next(row for row in sections if row["id"] == best_child[0])
{"child": best_child[0] + f":c{best_child[1]:02d}", "child_words": len(best_child[2].split()),
 "parent_heading": parent["heading"], "parent_words": len(parent["text"].split())}

### Check it yourself

Change `size` to 40 and 400 and overlap to 0 and 40. Does top-1 change? When would you cite a child, and when a parent? In the notes, also record the number of embeddings and the total number of words returned to the model.